# EMV Motors — Warranty Cost Intelligence

## Stage 1 — Plan

## Project
Cost Reduction of claims in FY 2025-26

## Purpose of this notebook
Generates synthetic warranty claims dataset across three relational tables for downstream analysis.

## Problem statement
EMV Motors is facing increased warranty expenditures due to increasing  warranty claims over  the fiscal year 2025-2026. This increased cost is impacting  QA/QC teams and R&D teams budgets. This analysis will identify the highest-cost failure modes and components, enabling the Engineering team to prioritise mitigation strategy and resource allocation and target a reduction of 10% in warranty expenditure by Q3 2026.

### Analytical Questions
1. Which components generate the highest warranty cost, and at what claim rate per unit sold?
2. Which use cases report the most claims across geographic markets in FY 2025-26?
3. Does supplier quality score correlate with claim cost?
4. What is the cost of claims arising within 6 and 12 months of sale?
5. Has total warranty claim cost increased across FY 2025-26, and by how much?

## Author
Shubham Yadav| 03-04-2026

## Stage 2 — Analyse: Data Generation

### Table 1 — Suppliers

In [101]:
import numpy as np
import pandas as pd
import random
import string
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

In [102]:
np.random.seed(42)
random.seed(42)

In [103]:
def generate_code():
    chars = string.ascii_uppercase + string.digits
    part1 = ''.join(random.choice(chars) for _ in range(4))
    part2 = ''.join(random.choice(chars) for _ in range(4))
    return f"{part1}-{part2}"

unique_codes = set()

while len(unique_codes) < 8:
    unique_codes.add(generate_code())

for code in unique_codes:
    print(code)

AK1V-RJNV
HBRP-OIG8
F1CB-FNO6
GFYG-WWQC
38HY-F9SX
B9M8-0O2R
MECO-SFOG
YR3X-KXWN


In [104]:
df = pd.DataFrame({"supplier_code": list(unique_codes)})

print(df)

  supplier_code
0     AK1V-RJNV
1     HBRP-OIG8
2     F1CB-FNO6
3     GFYG-WWQC
4     38HY-F9SX
5     B9M8-0O2R
6     MECO-SFOG
7     YR3X-KXWN


In [105]:
values = [5, 6, 7, 8, 9, 10]
df['quality_score'] = np.random.choice(values, size=8, p=[0.133, 0.133, 0.133, 0.2, 0.2, 0.201])
print(df)

  supplier_code  quality_score
0     AK1V-RJNV              7
1     HBRP-OIG8             10
2     F1CB-FNO6              9
3     GFYG-WWQC              8
4     38HY-F9SX              6
5     B9M8-0O2R              6
6     MECO-SFOG              5
7     YR3X-KXWN             10


In [106]:
conditions = [
    df['quality_score'] <= 5,
    (df['quality_score'] > 5) & (df['quality_score'] < 8),
    df['quality_score'] >= 8
]
labels = ['Poor', 'Good', 'Excellent']
df['quality_band'] = np.select(conditions, labels, default='Unknown')
print(df)

  supplier_code  quality_score quality_band
0     AK1V-RJNV              7         Good
1     HBRP-OIG8             10    Excellent
2     F1CB-FNO6              9    Excellent
3     GFYG-WWQC              8    Excellent
4     38HY-F9SX              6         Good
5     B9M8-0O2R              6         Good
6     MECO-SFOG              5         Poor
7     YR3X-KXWN             10    Excellent


In [107]:
tiers = ['Tier 1'] * 4 + ['Tier 2'] * 4
np.random.shuffle(tiers)
df['tier'] = tiers
print(df)

  supplier_code  quality_score quality_band    tier
0     AK1V-RJNV              7         Good  Tier 1
1     HBRP-OIG8             10    Excellent  Tier 2
2     F1CB-FNO6              9    Excellent  Tier 2
3     GFYG-WWQC              8    Excellent  Tier 1
4     38HY-F9SX              6         Good  Tier 2
5     B9M8-0O2R              6         Good  Tier 2
6     MECO-SFOG              5         Poor  Tier 1
7     YR3X-KXWN             10    Excellent  Tier 1


In [108]:
cities=['Beijing', 'Manesar', 'Bangalore', 'Paris', 'Berlin', 'Mexico City', 'Dubai', 'Abu Dhabi']
df['city']=random.sample(cities, len(df))
print(df)

  supplier_code  quality_score quality_band    tier         city
0     AK1V-RJNV              7         Good  Tier 1       Berlin
1     HBRP-OIG8             10    Excellent  Tier 2  Mexico City
2     F1CB-FNO6              9    Excellent  Tier 2        Dubai
3     GFYG-WWQC              8    Excellent  Tier 1      Beijing
4     38HY-F9SX              6         Good  Tier 2      Manesar
5     B9M8-0O2R              6         Good  Tier 2    Bangalore
6     MECO-SFOG              5         Poor  Tier 1    Abu Dhabi
7     YR3X-KXWN             10    Excellent  Tier 1        Paris


In [109]:
start_date = pd.to_datetime("2020-01-01")
end_date = pd.to_datetime("2025-12-31")

delta_days = (end_date - start_date).days
df["contract_start_date"] = start_date + pd.to_timedelta(np.random.randint(0, delta_days + 1, size=len(df)),
    unit="D")
print(df)

  supplier_code  quality_score quality_band    tier         city  \
0     AK1V-RJNV              7         Good  Tier 1       Berlin   
1     HBRP-OIG8             10    Excellent  Tier 2  Mexico City   
2     F1CB-FNO6              9    Excellent  Tier 2        Dubai   
3     GFYG-WWQC              8    Excellent  Tier 1      Beijing   
4     38HY-F9SX              6         Good  Tier 2      Manesar   
5     B9M8-0O2R              6         Good  Tier 2    Bangalore   
6     MECO-SFOG              5         Poor  Tier 1    Abu Dhabi   
7     YR3X-KXWN             10    Excellent  Tier 1        Paris   

  contract_start_date  
0          2023-04-30  
1          2022-08-13  
2          2023-03-30  
3          2021-04-04  
4          2020-01-22  
5          2022-01-17  
6          2021-04-19  
7          2022-12-18  


In [110]:
# Table 1 - Supplier details 
df.to_csv('suppliers.csv', index=False)

### Table 2 — Components & Sales

In [111]:
tier1_suppliers = df.loc[df["tier"] == "Tier 1", "supplier_code"]
print(tier1_suppliers)

0    AK1V-RJNV
3    GFYG-WWQC
6    MECO-SFOG
7    YR3X-KXWN
Name: supplier_code, dtype: object


In [112]:
tier1 = df[df['tier'] == 'Tier 1']['supplier_code'].tolist()
components = [
    {'name': 'Turbocharger', 'family': 'Powertrain', 'primary_supplier': tier1[0]},
    {'name': 'Brake Caliper', 'family': 'Chassis', 'primary_supplier': tier1[0]},
    {'name': 'Transmission Assembly', 'family': 'Powertrain', 'primary_supplier': tier1[0]},
    {'name': 'Drive Shaft', 'family': 'Powertrain', 'primary_supplier': tier1[0]},
    {'name': 'Alternator', 'family': 'Electrical', 'primary_supplier': tier1[1]},
    {'name': 'Starter Motor', 'family': 'Electrical', 'primary_supplier':  tier1[1]},
    {'name': 'Blower Motor', 'family': 'HVAC', 'primary_supplier':  tier1[1]},
    {'name': 'Compressor', 'family': 'HVAC', 'primary_supplier':  tier1[1]},
    {'name': 'Steering Knuckle', 'family': 'Chassis', 'primary_supplier':  tier1[2]},
    {'name': 'Tie Rod', 'family': 'Chassis', 'primary_supplier':  tier1[2]},
    {'name': 'Steering Column', 'family': 'Chassis', 'primary_supplier':  tier1[2]},
    {'name': 'Suspension Control Arm', 'family': 'Chassis', 'primary_supplier':  tier1[2]},
    {'name': 'Door Panel', 'family': 'Body', 'primary_supplier':  tier1[3]},
    {'name': 'Front Bumper', 'family': 'Body', 'primary_supplier': tier1[3]},
    {'name': 'Tailgate', 'family': 'Body', 'primary_supplier': tier1[3]},
]
print(f"Total components: {len(components)}")

Total components: 15


In [113]:
def generate_component_code():
    chars = string.ascii_uppercase + string.digits
    part1 = ''.join(random.choice(chars) for _ in range(3))
    part2 = ''.join(random.choice(chars) for _ in range(4))
    return f"{part1}-{part2}"

unique_component_codes = set()

while len(unique_component_codes) < 15:
    unique_component_codes.add(generate_component_code())

for code in unique_component_codes:
    print(code)

RVH-S1K3
AQ6-L6GT
IP9-8Q1Z
HJK-1EYY
71N-8MTZ
37Q-9AH8
8II-49KQ
6MJ-XK87
3YR-9OUD
X27-2HPO
PDP-FF5E
AU5-BHXT
OCU-ZREN
UN5-Z3JQ
XOI-65FD


In [114]:
component_codes = list(unique_component_codes)
for i, component in enumerate(components):component['component_code'] = component_codes[i]
for c in components[:3]:
    print(c)

{'name': 'Turbocharger', 'family': 'Powertrain', 'primary_supplier': 'AK1V-RJNV', 'component_code': 'RVH-S1K3'}
{'name': 'Brake Caliper', 'family': 'Chassis', 'primary_supplier': 'AK1V-RJNV', 'component_code': 'AQ6-L6GT'}
{'name': 'Transmission Assembly', 'family': 'Powertrain', 'primary_supplier': 'AK1V-RJNV', 'component_code': 'IP9-8Q1Z'}


In [115]:
df_components = pd.DataFrame(components)
print(df_components)

                      name      family primary_supplier component_code
0             Turbocharger  Powertrain        AK1V-RJNV       RVH-S1K3
1            Brake Caliper     Chassis        AK1V-RJNV       AQ6-L6GT
2    Transmission Assembly  Powertrain        AK1V-RJNV       IP9-8Q1Z
3              Drive Shaft  Powertrain        AK1V-RJNV       HJK-1EYY
4               Alternator  Electrical        GFYG-WWQC       71N-8MTZ
5            Starter Motor  Electrical        GFYG-WWQC       37Q-9AH8
6             Blower Motor        HVAC        GFYG-WWQC       8II-49KQ
7               Compressor        HVAC        GFYG-WWQC       6MJ-XK87
8         Steering Knuckle     Chassis        MECO-SFOG       3YR-9OUD
9                  Tie Rod     Chassis        MECO-SFOG       X27-2HPO
10         Steering Column     Chassis        MECO-SFOG       PDP-FF5E
11  Suspension Control Arm     Chassis        MECO-SFOG       AU5-BHXT
12              Door Panel        Body        YR3X-KXWN       OCU-ZREN
13    

In [116]:
df_sales = pd.DataFrame({
    'component_code': np.random.choice(df_components['component_code'], size=13500)
})
print(df_sales.shape)

(13500, 1)


In [117]:
df_sales["sale_id"] = [f"SALE-{i:05d}" for i in range(1, 13501)]
df_sales.shape

(13500, 2)

In [118]:
df_sales["sale_date"] = pd.to_datetime("2024-10-01") + pd.to_timedelta(
    np.random.randint(0, (pd.to_datetime("2026-03-31") - pd.to_datetime("2024-10-01")).days + 1, 13500),
    unit="D"
)
print(df_sales['sale_date'].min(), df_sales['sale_date'].max())


2024-10-01 00:00:00 2026-03-31 00:00:00


In [119]:
df_sales["use_case"] = np.random.choice(["Commuting", "Commercial", "Leisure", "Off-roading"], 13500)
df_sales["market_region"] = np.random.choice(["UK", "EU", "MENA", "Americas"], 13500)
df_sales.shape

(13500, 5)

In [120]:
tier2_suppliers = df.loc[df["tier"] == "Tier 2", "supplier_code"]
print(tier2_suppliers)

1    HBRP-OIG8
2    F1CB-FNO6
4    38HY-F9SX
5    B9M8-0O2R
Name: supplier_code, dtype: object


In [121]:
df_sales = df_sales.merge(df_components[["component_code", "primary_supplier"]], on="component_code", how="left")
print(df_sales.shape)
print(df_sales.head(3))

(13500, 6)
  component_code     sale_id  sale_date     use_case market_region  \
0       XOI-65FD  SALE-00001 2026-01-15  Off-roading            EU   
1       X27-2HPO  SALE-00002 2026-01-25    Commuting          MENA   
2       AU5-BHXT  SALE-00003 2024-10-23    Commuting            EU   

  primary_supplier  
0        YR3X-KXWN  
1        MECO-SFOG  
2        MECO-SFOG  


In [122]:
tier2_supplier_list = tier2_suppliers.tolist()

df_sales["supplier_code"] = np.where(
    np.random.rand(13500) <= 0.75,
    df_sales["primary_supplier"],
    np.random.choice(tier2_supplier_list, 13500)
)
print(df_sales['supplier_code'].value_counts())

supplier_code
AK1V-RJNV    2745
GFYG-WWQC    2677
MECO-SFOG    2658
YR3X-KXWN    2029
HBRP-OIG8     858
38HY-F9SX     854
F1CB-FNO6     849
B9M8-0O2R     830
Name: count, dtype: int64


In [123]:
#supplier row counts are proportional to component portfolio size, not supplier performance here.

In [124]:
df_sales = df_sales.drop(columns=['primary_supplier'])
print(df_sales.columns.tolist())

['component_code', 'sale_id', 'sale_date', 'use_case', 'market_region', 'supplier_code']


In [125]:
df_sales = df_sales[["sale_id", "component_code", "supplier_code", "sale_date", "market_region", "use_case"]]
print(df_sales.head(3))

      sale_id component_code supplier_code  sale_date market_region  \
0  SALE-00001       XOI-65FD     YR3X-KXWN 2026-01-15            EU   
1  SALE-00002       X27-2HPO     MECO-SFOG 2026-01-25          MENA   
2  SALE-00003       AU5-BHXT     MECO-SFOG 2024-10-23            EU   

      use_case  
0  Off-roading  
1    Commuting  
2    Commuting  


In [126]:
print(f"Shape: {df_sales.shape}")
print(f"Date range: {df_sales['sale_date'].min()} to {df_sales['sale_date'].max()}")
print(f"Unique components: {df_sales['component_code'].nunique()}")
print(f"Unique suppliers: {df_sales['supplier_code'].nunique()}")
print(f"Use cases: {df_sales['use_case'].unique()}")
print(f"Regions: {df_sales['market_region'].unique()}")

Shape: (13500, 6)
Date range: 2024-10-01 00:00:00 to 2026-03-31 00:00:00
Unique components: 15
Unique suppliers: 8
Use cases: ['Off-roading' 'Commuting' 'Leisure' 'Commercial']
Regions: ['EU' 'MENA' 'Americas' 'UK']


### Table 3 — Claims

In [127]:
df_sales = df_sales[df_sales["sale_date"] + pd.Timedelta(days=548) <= pd.to_datetime("2026-12-31")]
print(df_sales.shape)
print(f"Latest eligible sale date: {df_sales['sale_date'].max()}")

(6812, 6)
Latest eligible sale date: 2025-07-01 00:00:00


In [128]:
df_claims = df_sales.sample(n=2000, random_state=42)
print(df_claims.shape)

(2000, 6)


In [129]:
df_claims = df_claims.reset_index(drop=True)
df_claims['claim_id'] = [f"CLM-{i:05d}" for i in range(1, 2001)]
print(df_claims.head(3))

      sale_id component_code supplier_code  sale_date market_region  \
0  SALE-08114       37Q-9AH8     GFYG-WWQC 2025-03-17          MENA   
1  SALE-05843       XOI-65FD     YR3X-KXWN 2025-05-16      Americas   
2  SALE-02736       3YR-9OUD     MECO-SFOG 2025-05-10            EU   

    use_case   claim_id  
0    Leisure  CLM-00001  
1  Commuting  CLM-00002  
2  Commuting  CLM-00003  


In [130]:
df_claims["claim_date"] = df_claims["sale_date"] + pd.to_timedelta(
    np.random.randint(1, 549, 2000), unit="D")
print(df_claims[['sale_date', 'claim_date']].head(5))
print(f"Max claim date: {df_claims['claim_date'].max()}")

   sale_date claim_date
0 2025-03-17 2026-03-04
1 2025-05-16 2025-12-14
2 2025-05-10 2025-11-11
3 2024-10-29 2025-04-18
4 2024-10-27 2025-06-16
Max claim date: 2026-12-29 00:00:00


#resolving backdate in row 3
print(df_claims[df_claims['claim_date'] > pd.to_datetime('2026-12-31')])

In [131]:
print(df_claims[df_claims['claim_date'] <= df_claims['sale_date']])

Empty DataFrame
Columns: [sale_id, component_code, supplier_code, sale_date, market_region, use_case, claim_id, claim_date]
Index: []


In [132]:
# generating next column - days_since_sale
df_claims["days_since_sale"] = (df_claims["claim_date"]-df_claims["sale_date"]).dt.days
print(df_claims[['sale_date', 'claim_date', 'days_since_sale']].head(3))

   sale_date claim_date  days_since_sale
0 2025-03-17 2026-03-04              352
1 2025-05-16 2025-12-14              212
2 2025-05-10 2025-11-11              185


In [133]:
failure_mode_map = {
    'Powertrain': {
        'modes': ['Non-start', 'Leak', 'Noise', 'Corrosion', 'Intermittent'],
        'probs': [0.35, 0.35, 0.10, 0.10, 0.10]
    },
    'Chassis': {
        'modes': ['Noise', 'Corrosion','Intermittent', 'Leak', 'Non-start'],
        'probs': [0.35, 0.35, 0.10, 0.10, 0.10]
    },
    'Electrical': {
        'modes': ['Intermittent', 'Non-start','Leak', 'Corrosion','Noise'],
        'probs': [0.35, 0.35, 0.10, 0.10, 0.10]
    },
    'Body': {
        'modes': ['Corrosion', 'Noise','Non-start','Intermittent','Leak'],
        'probs': [0.35, 0.35, 0.10, 0.10, 0.10]
    },
     'HVAC': {
        'modes': ['Leak', 'Intermittent','Noise', 'Corrosion', 'Non-start' ],
        'probs': [0.35, 0.35, 0.10, 0.10, 0.10]
    }
    
}

In [134]:
print(df_claims.columns.tolist())

['sale_id', 'component_code', 'supplier_code', 'sale_date', 'market_region', 'use_case', 'claim_id', 'claim_date', 'days_since_sale']


In [135]:
df_claims = df_claims.merge(df_components[["component_code", "family", "name"]],on="component_code",how="left")
print(df_claims.columns.tolist())
print(df_claims.head(3))

['sale_id', 'component_code', 'supplier_code', 'sale_date', 'market_region', 'use_case', 'claim_id', 'claim_date', 'days_since_sale', 'family', 'name']
      sale_id component_code supplier_code  sale_date market_region  \
0  SALE-08114       37Q-9AH8     GFYG-WWQC 2025-03-17          MENA   
1  SALE-05843       XOI-65FD     YR3X-KXWN 2025-05-16      Americas   
2  SALE-02736       3YR-9OUD     MECO-SFOG 2025-05-10            EU   

    use_case   claim_id claim_date  days_since_sale      family  \
0    Leisure  CLM-00001 2026-03-04              352  Electrical   
1  Commuting  CLM-00002 2025-12-14              212        Body   
2  Commuting  CLM-00003 2025-11-11              185     Chassis   

               name  
0     Starter Motor  
1          Tailgate  
2  Steering Knuckle  


In [136]:
def assign_failure_mode(family):
    modes = failure_mode_map[family]['modes']
    probs = failure_mode_map[family]['probs']
    return np.random.choice(modes, p=probs)

df_claims['failure_mode'] = df_claims['family'].apply(assign_failure_mode)

print(df_claims.groupby(['family', 'failure_mode']).size().unstack(fill_value=0))

failure_mode  Corrosion  Intermittent  Leak  Noise  Non-start
family                                                       
Body                167            39    37    112         45
Chassis             239            66    70    221         81
Electrical           27            88    25     28         92
HVAC                 27            87    89     32         26
Powertrain           58            37   120     44        143


In [137]:
cost_range_map = {
    "Powertrain": (300, 3500),
    "Electrical": (80, 900),
    "Chassis": (120, 1200),
    "Body": (50, 600),
    "HVAC": (150, 1000)
}

def assign_claim_cost(family):
    low, high = cost_range_map[family]
    return round(np.random.uniform(low, high), 2)

df_claims['claim_cost_gbp'] = df_claims['family'].apply(assign_claim_cost)

print(df_claims.groupby('family')['claim_cost_gbp'].mean().round(2))

family
Body           328.57
Chassis        658.22
Electrical     492.13
HVAC           558.46
Powertrain    1909.13
Name: claim_cost_gbp, dtype: float64


In [138]:
df_claims['repeat_claim'] = df_claims.duplicated(subset=['sale_id', 'component_code'], keep=False)
print(df_claims['repeat_claim'].value_counts())

repeat_claim
False    2000
Name: count, dtype: int64


In [139]:
#Hence repeat claim rate is zero here.

In [140]:
print(df_claims.columns.tolist())

['sale_id', 'component_code', 'supplier_code', 'sale_date', 'market_region', 'use_case', 'claim_id', 'claim_date', 'days_since_sale', 'family', 'name', 'failure_mode', 'claim_cost_gbp', 'repeat_claim']


In [141]:
df_claims = df_claims.rename(columns={
    'family': 'component_family',
    'name': 'component_name'
})
df_claims=df_claims[['claim_id','sale_id','component_code',
                     'component_name','component_family','supplier_code',
                     'sale_date','claim_date','days_since_sale','market_region',
                     'use_case','failure_mode','claim_cost_gbp','repeat_claim']]

In [142]:
print(type(df_claims))
print(df_claims.shape)

<class 'pandas.core.frame.DataFrame'>
(2000, 14)


In [143]:
print(df_claims.head(3))
print(df_claims.dtypes)

    claim_id     sale_id component_code    component_name component_family  \
0  CLM-00001  SALE-08114       37Q-9AH8     Starter Motor       Electrical   
1  CLM-00002  SALE-05843       XOI-65FD          Tailgate             Body   
2  CLM-00003  SALE-02736       3YR-9OUD  Steering Knuckle          Chassis   

  supplier_code  sale_date claim_date  days_since_sale market_region  \
0     GFYG-WWQC 2025-03-17 2026-03-04              352          MENA   
1     YR3X-KXWN 2025-05-16 2025-12-14              212      Americas   
2     MECO-SFOG 2025-05-10 2025-11-11              185            EU   

    use_case failure_mode  claim_cost_gbp  repeat_claim  
0    Leisure         Leak          164.02         False  
1  Commuting    Corrosion          190.70         False  
2  Commuting        Noise          544.77         False  
claim_id                    object
sale_id                     object
component_code              object
component_name              object
component_family          

In [144]:
df.to_csv('suppliers.csv', index=False)
df_components.to_csv('components.csv', index=False)
df_claims.to_csv('claims.csv', index=False)
df_sales.to_csv('sales.csv', index=False)